# fhir4ds.viewdef

SQL-on-FHIR v2 ViewDefinition generator. This notebook demonstrates how to define portable FHIR views and execute them in DuckDB using the Zero-ETL pattern.

In [ ]:
# Install the unified package
%pip install fhir4ds-v2

In [ ]:
import json
import fhir4ds
from fhir4ds.sources import ExistingTableSource

con = fhir4ds.create_connection()

# Create a table conforming to the clinical schema
con.execute("""
    CREATE TABLE raw_data (
        id VARCHAR, 
        resourceType VARCHAR, 
        resource JSON, 
        patient_ref VARCHAR
    )
""")

patient = {
    "resourceType": "Patient",
    "id": "p1",
    "name": [
        {"given": ["John"], "family": "Doe"},
        {"given": ["Jane"], "family": "Smith"},
    ],
}
con.execute("INSERT INTO raw_data VALUES ('p1', 'Patient', ?, 'p1')", [json.dumps(patient)])

# Mount as the 'resources' view
fhir4ds.attach(con, ExistingTableSource("raw_data"))

### Collection Flattening with `forEach` 
ViewDefinitions can flatten arrays into rows using `forEach`. In DuckDB, this translates to efficient `CROSS JOIN LATERAL` / `UNNEST` operations.

In [ ]:
view_definition = {
    "resource": "Patient",
    "select": [
        {"column": [{"path": "id", "name": "patient_id"}]},
        {
            "forEach": "name",
            "column": [
                {"path": "family", "name": "family_name"},
                {"path": "given.first()", "name": "given_name"}
            ]
        }
    ],
}

sql = fhir4ds.generate_view_sql(view_definition)
con.execute(sql).df()